**4**&nbsp;&nbsp;&nbsp;Name your Jupyter Notebook as:

`TASK4_<your name>_<centre number>_<index number>.ipynb`

A library currently keeps paper records about its members, books and the books loaned. The library wants to create a suitable database to store the data and to allow them to run searches for specific data. The database will have three tables: a table to store data about the books, a table about the members and a table about the loans. The fields in each table are:

`Book`:

- `BookID` – unique book number, for example, 1234
- `Title` – the book title
- `Genre` – the type of book, for example, Drama, Sci-fi, Classic.

`Member`:

- `MemberNumber` – member's unique number, for example, 634
- `FamilyName` – member's family name
- `GivenName` – member's given name.

`Loan`:

- `LoanID` – the loan's unique number, for example, 12
- `MemberNumber` – the member's unique number
- `BookID` – the unique book number
- `DateLoaned` – the date that the book was taken out by the member
- `Returned` – TRUE if the book has been returned, or FALSE if it has **not** been returned.

For each of the sub-tasks 4.1 to 4.3, add a comment statement at the beginning of the code using the hash symbol '#', to indicate the sub-task the program code belongs to, for example:

```
In [1]: #Task 4.1
        Program code

        Output:
```


**Task 4.1**

Write a Python program that uses SQL code to create the database `LIBRARY` with the three tables given. Define the primary and foreign keys for each table. **[6]**


In [1]:
#Task 4.1
import sqlite3
conn = sqlite3.connect("library.db")
conn.execute('''CREATE TABLE "Book" (
	"BookID"	INTEGER NOT NULL,
	"Title"	TEXT NOT NULL,
	"Genre"	TEXT NOT NULL,
	PRIMARY KEY("BookID"));''')
conn.execute('''CREATE TABLE "Member" (
	"MemberNumber"	INTEGER NOT NULL,
	"FamilyName"	TEXT NOT NULL,
	"GivenName"	TEXT,
	PRIMARY KEY("MemberNumber"));''')
conn.execute('''CREATE TABLE "Loan" (
	"LoanID"	INTEGER NOT NULL,
	"MemberNumber"	INTEGER NOT NULL,
	"BookID"	INTEGER NOT NULL,
	"DateLoaned"	TEXT NOT NULL,
	"Returned"	TEXT NOT NULL,
	PRIMARY KEY("LoanID"),
	FOREIGN KEY("MemberNumber") REFERENCES "Member"("MemberNumber"));''')
conn.close()


**Task 4.2**

The text files `BOOK.txt`, `MEMBER.txt` and `LOAN.txt` store the comma-separated values for each of the tables in the database.

Write a Python program to read in the data from each file and then store each item of data in the correct place in the database. **[5]**


In [5]:
#Task 4.2
import csv,sqlite3
conn = sqlite3.connect("library.db")
with open("BOOK.txt","r") as file1:
    books = file1.readlines()
with open("MEMBER.txt","r") as file2:
    members = file2.readlines()
with open("LOAN.txt","r") as file3:
    loans = file3.readlines()
for book in books:
    book = book.strip().split(",")
    conn.execute('''INSERT INTO Book VALUES (?,?,?)''',(book[0],book[1],book[2]))
for member in members:
    member = member.strip().split(",")
    conn.execute('''INSERT INTO Member VALUES (?,?,?)''',(member[0],member[1],member[2]))
for loan in loans:
    loan = loan.strip().split(",")
    conn.execute('''INSERT INTO Loan VALUES (?,?,?,?,?)''',(loan[0],loan[1],loan[2],loan[3],loan[4]))
conn.commit()
conn.close()


**Task 4.3**

Write a Python program to input a member's number and return the names of all the books that they have had out on loan, and whether each book has been returned. **[5]**

Test your program by running the application with the member number 200 **[2]**

Save your Jupyter Notebook.


In [10]:
#Task 4.3
member = input("What member number would you like to check? ")
conn = sqlite3.connect("library.db")
cursor = conn.cursor()
result = cursor.execute('''SELECT book.title,loan.Returned FROM (Member JOIN Loan ON Member.MemberNumber=Loan.MemberNumber) AS link JOIN book ON link.BookID = book.BookID WHERE Member.MemberNumber = (?)''',(member,))
for line in result:
    print(f"Book Name: {line[0]}, Returned: {line[1]}")
conn.close()


Book Name: Monkey puzzle, Returned: FALSE
Book Name: Contemplating Camelias, Returned: FALSE
Book Name: Sandy shores, Returned: FALSE
Book Name: Propogation, Returned: TRUE


**Task 4.4**

Write a Python program and the necessary files to create a web application, that displays the following data, about books that have **not** yet been returned:

- member's family name
- member's given name
- book title.

The program should return an HTML document that enables the web browser to display a table with the required data.

Save your Python program as:

`TASK_4_4_<your name>_<centre number>_<index number>.py`

with any additional files / subfolders in a folder named:

`TASK_4_4_<your name>_<centre number>_<index number>` **[6]**

Run the web application.

Save the webpage output as:

`TASK_4_4_<your name>_<centre number>_<index number>.html` **[2]**


In [11]:
#Task 4.4
from flask import *
import sqlite3
query = '''SELECT member.FamilyName,member.GivenName,book.Title
FROM (Member JOIN Loan ON Member.MemberNumber=Loan.MemberNumber) AS link
JOIN book ON link.BookID = book.BookID
WHERE loan.returned = 'FALSE' '''
app = Flask(__name__)
@app.route("/")
def index():
    conn = sqlite3.connect("library.db")
    cursor = conn.cursor()
    result = cursor.execute(query).fetchall()
    return render_template("index.html",unreturn=result)
conn.close()